In [3]:
%pip install python-dotenv
import datetime as dt
import json
from dataclasses import dataclass, asdict
from typing import Any, Dict, Optional

import numpy as np
import pandas as pd
import yfinance as yf
from groq import Groq
import re
from pathlib import Path
from datetime import date
from dotenv import load_dotenv

import os



  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# --- Transcript capture helpers (paste near imports/globals) ---
from pathlib import Path
import json
transcript = []

def record_call(agent_name: str, system_prompt: str, user_payload, model_response_raw: str):
    transcript.append({
        "agent": agent_name,
        "system_prompt": system_prompt,
        "user_payload": user_payload,
        "model_response_raw": model_response_raw
    })

# ensure outputs dir exists
Path("outputs").mkdir(exist_ok=True)


In [5]:
import time
import random

def safe_call(fn, *args, **kwargs):
    """Retry wrapper for Groq rate limits."""
    for attempt in range(5):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            msg = str(e)
            if "rate_limit" in msg or "429" in msg:
                sleep_time = 2 + random.random() * 2
                print(f"Rate limit hit. Sleeping {sleep_time:.1f}s...")
                time.sleep(sleep_time)
            else:
                raise
    raise RuntimeError("Too many rate limit retries")


In [6]:
def extract_json(text: str):
    """
    Extract the first valid JSON object or array from an LLM response.
    Handles: clean JSON, JSON wrapped in markdown fences, stray backslash
    escapes before punctuation (e.g. >\ or >\ produced by some models).
    """
    def try_parse(s: str):
        try:
            return json.loads(s)
        except json.JSONDecodeError:
            pass
        # Remove stray backslashes before punctuation that some LLMs emit
        # e.g.  >\ becomes >   or  \/ becomes /
        cleaned = re.sub(r'\\(?=[^"\\nrtbfu])', '', s)
        try:
            return json.loads(cleaned)
        except json.JSONDecodeError:
            pass
        return None

    # 1. Try the raw text as-is
    result = try_parse(text.strip())
    if result is not None:
        return result

    # 2. Strip markdown code fences  ```json ... ``` or ``` ... ```
    fenced = re.search(r'```(?:json)?\s*(\{.*?\}|\[.*?\])\s*```', text, re.DOTALL)
    if fenced:
        result = try_parse(fenced.group(1))
        if result is not None:
            return result

    # 3. Find the first { ... } or [ ... ] block (greedy)
    for pattern in (r'\{.*\}', r'\[.*\]'):
        match = re.search(pattern, text, re.DOTALL)
        if match:
            result = try_parse(match.group())
            if result is not None:
                return result

    raise ValueError("LLM did not return valid JSON:\n" + text)


<>:5: SyntaxWarning: invalid escape sequence '\ '
<>:5: SyntaxWarning: invalid escape sequence '\ '
C:\Users\bholt\AppData\Local\Temp\ipykernel_5692\3865855618.py:5: SyntaxWarning: invalid escape sequence '\ '
  escapes before punctuation (e.g. >\ or >\ produced by some models).


In [7]:
# enforce_schema removed — the report generator now handles
# missing/alternate key names gracefully with fallback lookups.


In [8]:
TICKER = "AAPL"
TECH_LOOKBACK_DAYS = 365
FUND_LOOKBACK_YEARS = 10

today = dt.date.today()
tech_start = today - dt.timedelta(days=TECH_LOOKBACK_DAYS)
fund_start = today - dt.timedelta(days=FUND_LOOKBACK_YEARS * 365)

# Load variables from the .env file into the system environment
load_dotenv()

# Securely extract the API key
api_key = os.environ.get("GROQ_API_KEY")

# Initialize your client safely
client = Groq(api_key=api_key)

In [9]:
today = dt.date.today()
tech_start = today - dt.timedelta(days=TECH_LOOKBACK_DAYS)
fund_start = today - dt.timedelta(days=FUND_LOOKBACK_YEARS * 365)

In [10]:
TECHNICAL_SYSTEM_PROMPT = """
You are a Technical Analyst specializing in short-term price action and reversal setups.

Inputs you will receive:
- OHLCV data
- Moving averages (20, 50, 200)
- RSI(14)
- ATR(14)
- Volume trends
- Recent key levels (20-day high/low)

Your job:
1. Produce EXACTLY 3 structured points:
   - Trend
   - Momentum
   - Key Levels
2. Give a directional view: Bullish / Bearish / Neutral
3. Give a time horizon: 3–10 days
4. Give an invalidation level (where the thesis breaks)
5. Suggest a position size: Full / Half / Watchlist / Avoid

Constraints:
- You are a short-term trader.
- You do NOT consider fundamentals.
- You must be concise and structured.

IMPORTANT — you MUST use EXACTLY these top-level JSON keys, no others:
{
  "analysis": [{"point": "Trend", "detail": "..."}, {"point": "Momentum", "detail": "..."}, {"point": "Key Levels", "detail": "..."}],
  "view": "<Bullish|Bearish|Neutral>",
  "horizon": "<e.g. 3-10 days>",
  "invalidation": "<price level or condition>",
  "position_size": "<Full|Half|Watchlist|Avoid>"
}
Return a valid JSON object with exactly those keys.
"""

FUNDAMENTAL_SYSTEM_PROMPT = """
You are a Fundamental Analyst specializing in long-term dividend-growth investing.

Inputs you will receive:
- Revenue growth
- EPS growth
- Dividend growth (3y, 5y)
- Payout ratio
- Debt/equity
- Free cash flow
- Valuation multiples (P/E, P/S)

Your job:
1. Produce EXACTLY 3 structured points:
   - Growth
   - Quality
   - Valuation
2. Give a directional view: Bullish / Bearish / Neutral
3. Give a time horizon: 6 to 24 months
4. Provide key upside drivers
5. Provide key downside risks
6. Suggest a position size: Full / Half / Watchlist / Avoid

Constraints:
- You are long-only.
- You prioritize dividend safety and consistency.
- You must be structured and concise.

IMPORTANT — you MUST use EXACTLY these top-level JSON keys, no others:
{
  "Growth": {"RevenueGrowth": "...", "EarningsGrowth": "...", "DividendGrowth": "..."},
  "Quality": {"PayoutRatio": "...", "FreeCashFlow": "...", "DebtToEquity": "..."},
  "Valuation": {"PE": "...", "PS": "...", "TargetPrice": "..."},
  "View": "<Bullish|Bearish|Neutral>",
  "TimeHorizonMonths": <integer>,
  "UpsideDrivers": ["...", "..."],
  "DownsideRisks": ["...", "..."],
  "PositionSize": "<Full|Half|Watchlist|Avoid>"
}
Return a valid JSON object with exactly those keys.
"""

SYNTH_SYSTEM_PROMPT = """
You are the Investment Committee Synthesizer.

Inputs you will receive:
- Full technical memo
- Full fundamental memo

Your job:
1. Identify agreement between the two analysts
2. Identify disagreement
3. Weigh evidence and risk
4. Produce:
   - Bull case
   - Bear case
   - Key risks
   - Final recommendation
   - Final position size

Constraints:
- You must consider BOTH time horizons.
- You must justify the final position size.
- You must be structured and concise.

IMPORTANT — you MUST use EXACTLY these top-level JSON keys, no others:
{
  "agreement": ["...", "..."],
  "disagreement": ["...", "..."],
  "bull_case": "...",
  "bear_case": "...",
  "key_risks": ["...", "..."],
  "final_recommendation": "...",
  "position_size": "..."
}
Return a valid JSON object with exactly those keys.
"""
CHALLENGER_PROMPT = """
Give up to 3 critiques. Each must include:
- title
- severity (H/M/L)
- fix
Return JSON list.
"""


REPLY_PROMPT = """
Reply to the critiques provided. For each critique:
- accept: true/false
- fix: the specific change made (if accepted), or null

Then return a "revised_memo" that is the original memo updated with only the accepted fixes.
The revised_memo must preserve ALL original keys. Only update keys where a fix was accepted.
Return JSON in exactly this format:
{
  "replies": [{"accept": true/false, "fix": "..." or null}, ...],
  "revised_memo": { ...complete updated memo... }
}
"""


QUESTIONER_PROMPT = """
Ask 3 to 5 short questions that expose conflicts or missing assumptions.
Return JSON array of strings.
"""
ONE_PAGE_PROMPT = (
    "Produce a one-page investment memo JSON with fields: "
    "Title, OneLineThesis, Top3UpsideDrivers, Top3DownsideRisks, PositionSize, ThreeLineRationale. "
    "Input: technical_memo and fundamental_memo. Return JSON only."
)







In [11]:
def pretty(obj: Any) -> None:
    print(json.dumps(obj, indent=4))

In [12]:
def fetch_ohlcv(ticker: str, start: dt.date, end: dt.date) -> pd.DataFrame:
    df = yf.download(ticker, start=start, end=end)

    # Normalize columns
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = ["_".join(col).lower() for col in df.columns]
    else:
        df.columns = [col.lower() for col in df.columns]

    # Handle ticker-suffixed columns like close_aapl
    rename_map = {
        f"open_{ticker.lower()}": "open",
        f"high_{ticker.lower()}": "high",
        f"low_{ticker.lower()}": "low",
        f"close_{ticker.lower()}": "close",
        f"volume_{ticker.lower()}": "volume",
    }
    df = df.rename(columns=rename_map)

    # If already standard, keep as is
    # Ensure we have needed columns
    required = {"open", "high", "low", "close", "volume"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required OHLCV columns: {missing}")

    return df[["open", "high", "low", "close", "volume"]]


def compute_technical_indicators(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Moving averages
    df["ma20"] = df["close"].rolling(20).mean()
    df["ma50"] = df["close"].rolling(50).mean()
    df["ma200"] = df["close"].rolling(200).mean()

    # ATR(14)
    high_low = df["high"] - df["low"]
    high_close = (df["high"] - df["close"].shift()).abs()
    low_close = (df["low"] - df["close"].shift()).abs()
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    df["atr14"] = tr.rolling(14).mean()

    # RSI(14) – stable version
    delta = df["close"].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()
    rs = avg_gain / (avg_loss + 1e-9)
    df["rsi14"] = 100 - (100 / (1 + rs))

    # 20-day average volume
    df["avg_vol20"] = df["volume"].rolling(20).mean()

    return df


def build_technical_snapshot(df: pd.DataFrame) -> Dict[str, Any]:
    df = df.dropna().copy()
    last = df.iloc[-1].to_dict()

    snapshot = {
        "ticker": TICKER,
        "price": float(last["close"]),
        "ma20": float(last["ma20"]),
        "ma50": float(last["ma50"]),
        "ma200": float(last["ma200"]),
        "rsi14": float(last["rsi14"]),
        "atr14": float(last["atr14"]),
        "avg_volume20": float(last["avg_vol20"]),
        "volume": float(last["volume"]),
        "recent_high_20": float(df["high"].tail(20).max()),
        "recent_low_20": float(df["low"].tail(20).min()),
    }
    return snapshot


def fetch_fundamentals(ticker: str) -> Dict[str, Optional[float]]:
    stock = yf.Ticker(ticker)
    info = stock.info
    dividends = stock.dividends

    div_growth_3y = None
    div_growth_5y = None

    if not dividends.empty:
        div_df = dividends.to_frame(name="div")
        div_df["year"] = div_df.index.year
        yearly = div_df.groupby("year")["div"].sum()

        def cagr(series: pd.Series, years: int) -> Optional[float]:
            if len(series) < years + 1:
                return None
            end = series.iloc[-1]
            start = series.iloc[-(years + 1)]
            if start <= 0:
                return None
            return (end / start) ** (1 / years) - 1

        div_growth_3y = cagr(yearly, 3)
        div_growth_5y = cagr(yearly, 5)

    fundamentals = {
        "ticker": ticker,
        "market_cap": info.get("marketCap"),
        "sector": info.get("sector"),
        "dividend_yield": info.get("dividendYield"),
        "payout_ratio": info.get("payoutRatio"),
        "pe_ratio": info.get("trailingPE"),
        "ps_ratio": info.get("priceToSalesTrailing12Months"),
        "debt_to_equity": info.get("debtToEquity"),
        "revenue_growth": info.get("revenueGrowth"),
        "earnings_growth": info.get("earningsGrowth"),
        "free_cash_flow": info.get("freeCashflow"),
        "dividend_growth_3y": div_growth_3y,
        "dividend_growth_5y": div_growth_5y,
    }
    return fundamentals

In [13]:
@dataclass
class TechnicalInput:
    ticker: str
    price: float
    ma20: float
    ma50: float
    ma200: float
    rsi14: float
    atr14: float
    avg_volume20: float
    volume: float
    recent_high_20: float
    recent_low_20: float


@dataclass
class FundamentalInput:
    ticker: str
    market_cap: Optional[float]
    sector: Optional[str]
    dividend_yield: Optional[float]
    payout_ratio: Optional[float]
    pe_ratio: Optional[float]
    ps_ratio: Optional[float]
    debt_to_equity: Optional[float]
    revenue_growth: Optional[float]
    earnings_growth: Optional[float]
    free_cash_flow: Optional[float]
    dividend_growth_3y: Optional[float]
    dividend_growth_5y: Optional[float]

In [14]:
def call_technical_agent(tech_input: TechnicalInput):
    def _call():
        payload = asdict(tech_input)
        resp = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[
                {"role": "system", "content": TECHNICAL_SYSTEM_PROMPT},
                {"role": "user", "content": json.dumps(payload)},
            ],
            temperature=0.2,
        )
        raw = resp.choices[0].message.content
        # record raw response before parsing
        record_call("technical_agent", TECHNICAL_SYSTEM_PROMPT, payload, raw)
        return extract_json(raw)

    result = safe_call(_call)
    time.sleep(0.3)
    return result


def call_fundamental_agent(fund_input: FundamentalInput):
    def _call():
        payload = asdict(fund_input)
        resp = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[
                {"role": "system", "content": FUNDAMENTAL_SYSTEM_PROMPT},
                {"role": "user", "content": json.dumps(payload)},
            ],
            temperature=0.2,
        )
        raw = resp.choices[0].message.content
        record_call("fundamental_agent", FUNDAMENTAL_SYSTEM_PROMPT, payload, raw)
        return extract_json(raw)

    result = safe_call(_call)
    time.sleep(0.3)
    return result


def call_synthesizer_agent(tech_memo, fund_memo):
    def _call():
        payload = {
            "technical_memo": tech_memo,
            "fundamental_memo": fund_memo,
        }
        resp = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[
                {"role": "system", "content": SYNTH_SYSTEM_PROMPT},
                {"role": "user", "content": json.dumps(payload)},
            ],
            temperature=0.2,
        )
        raw = resp.choices[0].message.content
        record_call("synthesizer_agent", SYNTH_SYSTEM_PROMPT, payload, raw)
        return extract_json(raw)

    result = safe_call(_call)
    time.sleep(0.3)
    return result


def call_challenger_agent(target_memo: dict, challenger_prompt: str):
    def _call():
        payload = {"target_memo": target_memo}
        resp = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[
                {"role": "system", "content": challenger_prompt},
                {"role": "user", "content": json.dumps(payload)},
            ],
            temperature=0.2,
        )
        raw = resp.choices[0].message.content
        record_call("challenger_agent", challenger_prompt, payload, raw)
        return extract_json(raw)

    result = safe_call(_call)
    time.sleep(0.3)
    return result


def call_agent_reply_agent(original_memo: dict, critiques: dict, reply_prompt: str):
    def _call():
        payload = {"original_memo": original_memo, "critiques": critiques}
        resp = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[
                {"role": "system", "content": reply_prompt},
                {"role": "user", "content": json.dumps(payload)},
            ],
            temperature=0.2,
        )
        raw = resp.choices[0].message.content
        record_call("agent_reply_agent", reply_prompt, payload, raw)
        return extract_json(raw)

    result = safe_call(_call)
    time.sleep(0.3)
    return result


def call_questioner(tech_memo: dict, fund_memo: dict, questioner_prompt: str):
    def _call():
        payload = {"technical_memo": tech_memo, "fundamental_memo": fund_memo}
        resp = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[
                {"role": "system", "content": questioner_prompt},
                {"role": "user", "content": json.dumps(payload)},
            ],
            temperature=0.2,
        )
        raw = resp.choices[0].message.content
        record_call("questioner_agent", questioner_prompt, payload, raw)
        return extract_json(raw)

    result = safe_call(_call)
    time.sleep(0.3)
    return result


def safe_merge_memo(original, updated):
    """Merge updated into original.
    Rules:
    - Skip keys whose value is None, empty string, empty list/dict, or "N/A"
    - Recursively merge nested dicts
    - Never discard a key that exists in original if the update has nothing useful
    """
    if not updated:
        return original
    merged = dict(original)
    for k, v in updated.items():
        # Skip empty / placeholder values — don't let them overwrite good data
        if v is None or v == "" or v == "N/A" or v == [] or v == {}:
            continue
        if isinstance(v, dict) and isinstance(original.get(k), dict):
            merged[k] = safe_merge_memo(original[k], v)
        else:
            merged[k] = v
    return merged






In [15]:
def generate_markdown_report(
    ticker: str,
    tech_memo: dict,
    fund_memo: dict,
    final_memo: dict,
    output_path: str | Path = None,
) -> Path:
    if output_path is None:
        output_path = Path(f"investment_report_{ticker}_{date.today()}.md")
    else:
        output_path = Path(output_path)

    t = tech_memo
    f = fund_memo
    s = final_memo

    def get(d, *keys, default="N/A"):
        """Try multiple key spellings; return first non-empty match."""
        for k in keys:
            v = d.get(k)
            if v is not None and v != "" and v != "N/A":
                return v
        return default

    def fmt(v):
        """Render a value: dicts as inline JSON, everything else as string."""
        if isinstance(v, dict):
            return json.dumps(v, ensure_ascii=False)
        if isinstance(v, list):
            return ", ".join(str(i) for i in v)
        return str(v)

    lines = []

    # ── Title ─────────────────────────────────────────────────────────────────
    lines += [f"# Investment Committee Report: {ticker}", "",
              f"_Date: {date.today()}_", ""]

    # ── Executive Summary ─────────────────────────────────────────────────────
    # Technical: view key is "view", horizon is "horizon"
    tech_view    = get(t, "view", "directionalView", "directional_view")
    tech_horizon = get(t, "horizon", "timeHorizon", "time_horizon", "time_horizon_days")
    # Fundamental: View (capital V), TimeHorizonMonths
    fund_view    = get(f, "View", "view", "directionalView", "directional_view")
    fund_horizon = get(f, "TimeHorizonMonths", "timeHorizon", "time_horizon", "time_horizon_months")
    # Final: position_size (not final_position_size)
    final_pos    = get(s, "final_position_size", "position_size", "finalPositionSize")
    final_rec    = get(s, "final_recommendation", "finalRecommendation")

    lines += ["## Executive Summary", ""]
    lines.append(f"- **Ticker:** {ticker}")
    lines.append(f"- **Technical View:** {tech_view} ({tech_horizon})")
    lines.append(f"- **Fundamental View:** {fund_view} ({fund_horizon} months)")
    lines.append(f"- **Final Position Size:** {fmt(final_pos)}")
    lines += ["", f"**Final Recommendation:** {final_rec}", ""]

    # ── Technical Analysis ────────────────────────────────────────────────────
    tech_pos   = get(t, "position_size", "positionSize")
    tech_inv   = get(t, "invalidation", "invalidationLevel", "invalidation_level")

    lines += ["## Technical Analysis", ""]
    lines.append(f"- **Direction:** {tech_view}")
    lines.append(f"- **Time Horizon:** {tech_horizon}")
    lines.append(f"- **Position Size:** {fmt(tech_pos)}")
    lines.append(f"- **Invalidation Level:** {tech_inv}")
    lines += ["", "### Key Points", ""]

    # LLM returns analysis as a list of {point, detail} objects
    analysis = t.get("analysis")
    if isinstance(analysis, list):
        for item in analysis:
            if isinstance(item, dict):
                point  = item.get("point", "")
                detail = item.get("detail", "")
                lines.append(f"- **{point}:** {detail}")
    elif isinstance(analysis, dict):
        for k, v in analysis.items():
            lines.append(f"- **{k}:** {fmt(v)}")
    else:
        # Fallback: look for individual keys
        for key in ["trend", "momentum", "keyLevels", "key_levels"]:
            if key in t:
                val = t[key]
                if isinstance(val, dict):
                    lines.append(f"- **{key}:**")
                    for sk, sv in val.items():
                        lines.append(f"  - **{sk}:** {sv}")
                else:
                    lines.append(f"- **{key}:** {val}")
    lines.append("")

    # ── Fundamental Analysis ──────────────────────────────────────────────────
    fund_pos = get(f, "PositionSize", "positionSize", "position_size")

    lines += ["## Fundamental Analysis", ""]
    lines.append(f"- **View:** {fund_view}")
    lines.append(f"- **Time Horizon:** {fund_horizon} months")
    lines.append(f"- **Position Size:** {fmt(fund_pos)}")
    lines.append("")

    # Growth / Quality / Valuation — PascalCase in LLM output
    for section in [("Growth", "growth"), ("Quality", "quality"), ("Valuation", "valuation")]:
        pascal, lower = section
        lines.append(f"### {pascal}")
        val = get(f, pascal, lower)
        if isinstance(val, dict):
            for sk, sv in val.items():
                lines.append(f"- **{sk}:** {fmt(sv)}")
        else:
            lines.append(f"- {val}")
        lines.append("")

    lines += ["### Upside Drivers", ""]
    for u in f.get("UpsideDrivers", f.get("upside_drivers", f.get("upsideDrivers", []))):
        lines.append(f"- {u}")
    lines.append("")

    lines += ["### Downside Risks", ""]
    for d in f.get("DownsideRisks", f.get("downside_risks", f.get("downsideRisks", []))):
        lines.append(f"- {d}")
    lines.append("")

    # ── Investment Committee Synthesis ────────────────────────────────────────
    lines += ["## Investment Committee Synthesis", ""]

    lines.append("### Agreement")
    agreement = s.get("agreement", [])
    if agreement:
        for item in agreement:
            lines.append(f"- {item}")
    else:
        lines.append("- *(No explicit agreement items returned by the synthesizer.)*")
    lines.append("")

    lines.append("### Disagreement")
    disagreement = s.get("disagreement", [])
    if disagreement:
        for item in disagreement:
            lines.append(f"- {item}")
    else:
        lines.append("- *(No explicit disagreement items returned by the synthesizer.)*")
    lines.append("")

    lines.append("### Bull Case")
    bull = s.get("bull_case", s.get("bullCase", {}))
    if isinstance(bull, dict):
        for k, v in bull.items():
            lines.append(f"**{k.capitalize()}:** {v}")
            lines.append("")
    else:
        lines.append(str(bull))
    lines.append("")

    lines.append("### Bear Case")
    bear = s.get("bear_case", s.get("bearCase", {}))
    if isinstance(bear, dict):
        for k, v in bear.items():
            lines.append(f"**{k.capitalize()}:** {v}")
            lines.append("")
    else:
        lines.append(str(bear))
    lines.append("")

    lines += ["### Key Risks"]
    for r in s.get("key_risks", s.get("keyRisks", [])):
        lines.append(f"- {r}")
    lines.append("")

    lines += ["### Final Recommendation"]
    lines.append(f"- **Summary:** {final_rec}")
    lines.append("")

    lines += ["### Final Position Size"]
    lines.append(fmt(final_pos))
    lines.append("")

    # ── Appendix ──────────────────────────────────────────────────────────────
    lines += ["## Appendix", ""]
    for label, memo in [("Technical", t), ("Fundamental", f), ("Final IC", s)]:
        lines.append(f"### Raw {label} Memo")
        lines += ["", "```json", json.dumps(memo, indent=4, ensure_ascii=False), "```", ""]

    output_path.write_text("\n".join(lines), encoding="utf-8")
    return output_path


In [16]:
# Agent-to-agent conversation labels — maps record_call agent_name to a
# human-readable role and the logical pipeline stage it belongs to.
AGENT_LABELS = {
    "technical_agent":       ("📈 Technical Analyst",      "Round 1 — Initial Analysis"),
    "fundamental_agent":     ("📊 Fundamental Analyst",    "Round 1 — Initial Analysis"),
    "agent_reply_agent":     ("🔄 Reply Agent",            "Round 2 — Self-Check / Revision"),
    "challenger_agent":      ("⚔️  Challenger",             "Round 3 — Cross-Challenge"),
    "questioner_agent":      ("❓ Questioner",              "Round 4 — Question Round"),
    "synthesizer_agent":     ("🏛️  IC Synthesizer",         "Round 5 — Final Synthesis"),
    "one_page_generator":    ("📄 One-Page Generator",      "Round 6 — Summary"),
}

def generate_transcript_report(
    ticker: str,
    transcript: list,
    output_path: str | Path = None,
) -> Path:
    """
    Render the full agent-to-agent conversation transcript as a readable
    Markdown file. Each call becomes a collapsible section showing:
      - which agent spoke
      - what it received (user payload)
      - what it replied (raw model response)
    The system prompt for each agent is shown once per unique agent.
    """
    if output_path is None:
        output_path = Path(f"transcript_{ticker}_{date.today()}.md")
    else:
        output_path = Path(output_path)

    lines = []

    # Title
    lines += [
        f"# Agent Conversation Transcript: {ticker}",
        "",
        f"_Date: {date.today()}_",
        "",
        f"_Total agent calls: {len(transcript)}_",
        "",
        "---",
        "",
    ]

    # Index / table of contents
    lines += ["## Table of Contents", ""]
    for idx, entry in enumerate(transcript, 1):
        agent = entry.get("agent", "unknown")
        role, stage = AGENT_LABELS.get(agent, (agent, ""))
        lines.append(f"{idx}. [{role} — {stage}](#{idx}-{agent.replace('_', '-')})")
    lines += ["", "---", ""]

    # One section per call
    shown_system_prompts = set()
    for idx, entry in enumerate(transcript, 1):
        agent      = entry.get("agent", "unknown")
        sys_prompt = entry.get("system_prompt", "")
        payload    = entry.get("user_payload", {})
        raw_resp   = entry.get("model_response_raw", "")

        role, stage = AGENT_LABELS.get(agent, (agent, ""))

        lines += [
            f"## {idx}. {role}",
            f"**Stage:** {stage}  ",
            f"**Agent ID:** `{agent}`",
            "",
        ]

        # Show system prompt once per unique agent type
        if agent not in shown_system_prompts and sys_prompt:
            shown_system_prompts.add(agent)
            lines += [
                "<details>",
                "<summary><strong>System Prompt (click to expand)</strong></summary>",
                "",
                "```",
                sys_prompt.strip(),
                "```",
                "",
                "</details>",
                "",
            ]

        # User payload
        lines += ["### Input (User Payload)", ""]
        if isinstance(payload, dict):
            # Pretty-print but truncate large nested memos to keep it readable
            def truncate(obj, max_str=300):
                if isinstance(obj, str) and len(obj) > max_str:
                    return obj[:max_str] + "…"
                if isinstance(obj, dict):
                    return {k: truncate(v) for k, v in obj.items()}
                if isinstance(obj, list):
                    return [truncate(i) for i in obj]
                return obj
            lines += [
                "```json",
                json.dumps(truncate(payload), indent=2, ensure_ascii=False),
                "```",
                "",
            ]
        else:
            lines += [str(payload), ""]

        # Model response
        lines += ["### Output (Model Response)", ""]
        # Try to pretty-print if it looks like JSON, else show raw
        stripped = raw_resp.strip() if raw_resp else ""
        try:
            parsed = json.loads(stripped)
            lines += [
                "```json",
                json.dumps(parsed, indent=2, ensure_ascii=False),
                "```",
            ]
        except Exception:
            lines += [
                "```",
                stripped,
                "```",
            ]
        lines += ["", "---", ""]

    output_path.write_text("\n".join(lines), encoding="utf-8")
    return output_path


In [ ]:
def main() -> None:
    # Technical pipeline
    ohlcv = fetch_ohlcv(TICKER, tech_start, today)
    print("\nDEBUG — Raw columns:", ohlcv.columns.tolist())

    ohlcv = compute_technical_indicators(ohlcv)
    tech_snapshot = build_technical_snapshot(ohlcv)
    tech_input = TechnicalInput(**tech_snapshot)

    # Fundamental pipeline
    fund_raw = fetch_fundamentals(TICKER)
    fund_input = FundamentalInput(**fund_raw)

    print("\n=== RAW TECHNICAL VALUES ===")
    for k, v in tech_snapshot.items():
        print(f"{k}: {v}")

    print("\n=== RAW FUNDAMENTAL VALUES ===")
    for k, v in fund_raw.items():
        print(f"{k}: {v}")

    # --- FUNDAMENTAL: two rounds ---
    fund_v1 = call_fundamental_agent(fund_input)
    fund_v2 = call_agent_reply_agent(fund_v1, {"self_check": True}, REPLY_PROMPT)
    fund_memo = safe_merge_memo(fund_v1, fund_v2.get("revised_memo", {}))

    # --- TECHNICAL: two rounds ---
    tech_v1 = call_technical_agent(tech_input)
    tech_v2 = call_agent_reply_agent(tech_v1, {"self_check": True}, REPLY_PROMPT)
    tech_memo = safe_merge_memo(tech_v1, tech_v2.get("revised_memo", {}))

    # --- CROSS-CHALLENGE ---
    crit_by_fund = call_challenger_agent(tech_memo, CHALLENGER_PROMPT)
    crit_by_tech = call_challenger_agent(fund_memo, CHALLENGER_PROMPT)

    # --- REPLIES ---
    fund_reply = call_agent_reply_agent(fund_memo, crit_by_tech, REPLY_PROMPT)
    fund_memo = safe_merge_memo(fund_memo, fund_reply.get("revised_memo", {}))

    tech_reply = call_agent_reply_agent(tech_memo, crit_by_fund, REPLY_PROMPT)
    tech_memo = safe_merge_memo(tech_memo, tech_reply.get("revised_memo", {}))

    # --- QUESTION ROUND ---
    questions = call_questioner(tech_memo, fund_memo, QUESTIONER_PROMPT)

    fund_answers = call_agent_reply_agent(fund_memo, {"questions": questions}, REPLY_PROMPT)
    tech_answers = call_agent_reply_agent(tech_memo, {"questions": questions}, REPLY_PROMPT)

    fund_memo = safe_merge_memo(fund_memo, fund_answers.get("revised_memo", {}))
    tech_memo = safe_merge_memo(tech_memo, tech_answers.get("revised_memo", {}))

    # --- FINAL SYNTHESIS ---
    final_memo = call_synthesizer_agent(tech_memo, fund_memo)


    # --- Persist transcript and raw memos ---
    Path("outputs").mkdir(exist_ok=True)
    Path("outputs/transcript.json").write_text(json.dumps(transcript, indent=2), encoding="utf-8")
    Path("outputs/tech_memo_raw.json").write_text(json.dumps(tech_memo, indent=2), encoding="utf-8")
    Path("outputs/fund_memo_raw.json").write_text(json.dumps(fund_memo, indent=2), encoding="utf-8")
    Path("outputs/final_memo.json").write_text(json.dumps(final_memo, indent=2), encoding="utf-8")

    # Pretty output
    print("\n=== Technical Agent Memo ===")
    pretty(tech_memo)

    print("\n=== Fundamental Agent Memo ===")
    pretty(fund_memo)

    print("\n=== Final Investment Committee Memo ===")
    pretty(final_memo)

    report_path = generate_markdown_report(
        TICKER,
        tech_memo,
        fund_memo,
        final_memo
    )
    print(f"\nMarkdown report written to: {report_path}")

    transcript_path = generate_transcript_report(TICKER, transcript)
    print(f"Transcript report written to: {transcript_path}")

    # --- ONE-PAGE MEMO (NOW tech_memo & fund_memo EXIST) ---
    one_page_resp = safe_call(lambda: client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role":"system","content":ONE_PAGE_PROMPT},
            {"role":"user","content": json.dumps({
                "technical_memo": tech_memo,
                "fundamental_memo": fund_memo
            })}
        ],
        temperature=0.2,
    ))
    one_page_raw = one_page_resp.choices[0].message.content
    record_call("one_page_generator", ONE_PAGE_PROMPT,
                {"technical_memo":"saved","fundamental_memo":"saved"},
                one_page_raw)
    one_page_json = extract_json(one_page_raw)
    Path("outputs/one_page_memo.json").write_text(
        json.dumps(one_page_json, indent=2),
        encoding="utf-8"
    )

    # --- DONE. Reflection removed entirely. ---


if __name__ == "__main__":
    main()


[*********************100%***********************]  1 of 1 completed



DEBUG — Raw columns: ['open', 'high', 'low', 'close', 'volume']

=== RAW TECHNICAL VALUES ===
ticker: AAPL
price: 311.0
ma20: 323.72149963378905
ma50: 309.6542004394531
ma200: 278.5679763793945
rsi14: 35.64238956403305
atr14: 9.70357186453683
avg_volume20: 56408620.0
volume: 49178700.0
recent_high_20: 344.57000732421875
recent_low_20: 300.0

=== RAW FUNDAMENTAL VALUES ===
ticker: AAPL
market_cap: 4538790051840
sector: Technology
dividend_yield: 0.35
payout_ratio: 0.1204
pe_ratio: 35.706085
ps_ratio: 9.722722
debt_to_equity: 78.445
revenue_growth: 0.164
earnings_growth: 0.287
free_cash_flow: 107721875456
dividend_growth_3y: -0.17677714069774086
dividend_growth_5y: -0.09332434977885196
